# T2b · Scoring the two writers by the rule fixed before the run (your PC, CPU)

Reads the saved run of [t2b_writer_test.ipynb](t2b_writer_test.ipynb) (ten scenes, six reports each, written once
by our 7B and once by Qwen2.5-3B-Instruct, with the same facts, flow and settings) and runs no model. What T2b is,
why, the scenes and the decoding are in that notebook's first cell.

**The rule, fixed before the run** (roadmap T2b): the text model replaces the 7B as the writer only if, across all
60 reports, it has **no more contradictions**, **no more fake citations**, and **finishes at least as many reports**.
A tie goes to the smaller model. A swap also needs the supervisor's agreement (our goals name our VLM), and the six
benchmark tasks stay with the VLM either way.

**How each count is made, and why:**
- **Contradictions:** numbers that T2 step 7's check (rule 2: *wrong* or *misused*) flags in what the writer left
  after the flow asked it again. These are read from the flow's own answer checks (`problems_left` in
  `llm_calls.jsonl`); the last guard replaced them in the finished text. The finished reports are checked too, and
  anything flagged there is added. *Why not the finished reports alone:* the guard (20 Nov) replaces every flagged
  number, so every writer would score 0 by construction. *Why not the first answers:* both writers get the same
  second chance, and the reports are what the flow keeps; first answers are shown beside it.
- **Unsourced numbers** (in neither the facts nor the passages) are counted beside contradictions, as in step 7.
- **Fake citations:** bracketed guideline IDs in the finished reports that aren't in the index
  (`guideline_chunks.jsonl`). IDs that are real but weren't shown to that report (neither found by its searches nor
  in the earlier reports it was given) are counted beside it.
- **Finished:** the report has all its sections, none of them an error, and its scene passes the five pass rules.
- **Beside the rule (not deciding it):** first-answer number problems, sections asked again, real citations,
  figures the writer itself wrote as "not available offline" (the guard's own insertions left out), words per
  section, minutes per scene and the model's memory. An AI
  judge or a reading rubric grades wording but doesn't check numbers exactly, so reading stays secondary: five
  pairs of reports are written out for a blind read by a person (`blind/`).

The code-written grade section is skipped, as in step 7. Each report's sources are rebuilt from the run's own logs
the same way as in step 7 (the damage facts and the code's map reading from `run_log.txt`; the passages from the
search IDs in `llm_calls.jsonl`).

In [1]:
# ── Cell 1: The run, its writers, and the guideline index ─────────────────────────
import json
import random
import re
import sys
from pathlib import Path

import pandas as pd

T2 = Path(r"D:\CSE499AB_project\Essential_task_delegation\Offline_RAG_KR2")
RUN = T2 / "results" / "t2b" / "t2b_full_internet_on"         # the unpacked zip of the full run
OUT = T2 / "results" / "t2b"
if not RUN.is_dir():                                            # before the full run: the smoke test
    RUN, OUT = T2 / "results" / "t2b_smoke" / "t2b_smoke_internet_on", T2 / "results" / "t2b_smoke"
sys.path.insert(0, str(T2 / "disasteller_offline"))
import number_check as nc

INFO = json.load(open(RUN / "t2b_notebook_info.json", encoding="utf-8"))
WRITERS = list(INFO["writers"])
GRADES_HEADING = "Disaster grade of each location"
EXPERT, ALERTS, EMERGENCY = "Expert_team_report.txt", "Alerts_team_report_for_social_media.txt", "Emergency_team_report.txt"
# From fixed_flow.REPORTS: the code's map reading, the EMS-98 sections, and the earlier reports each one is given
SPECS = {EXPERT: {"map": False, "ems98": True, "sections": 4, "uses": []},
         ALERTS: {"map": True, "ems98": False, "uses": [EXPERT]},
         EMERGENCY: {"map": True, "ems98": False, "uses": [EXPERT, ALERTS]},
         "Emergency_service_human_resource_allocation.txt": {"map": False, "ems98": False, "uses": [EXPERT, ALERTS, EMERGENCY]},
         "Public_community_report.txt": {"map": False, "ems98": False, "uses": [EXPERT, ALERTS, EMERGENCY]},
         "Reconstruction_plan.txt": {"map": False, "ems98": True, "uses": [EXPERT, ALERTS, EMERGENCY]}}
GUARD = nc.GUARD                                            # "(number not available offline)", written by the guard
CHUNKS = {c["id"]: c for c in map(json.loads, open(T2 / "guideline_chunks.jsonl", encoding="utf-8"))}
ID_PREFIXES = ("EMS98-", "XBD-", "IWGSEM-", "INSARAG3-", "INSARAG2B-", "SPHERE-")
BRACKETED = re.compile(r"\[([A-Za-z0-9][A-Za-z0-9._-]*)\]")
NOT_AVAILABLE = re.compile(r"not available offline", re.IGNORECASE)


def passage(section_id):
    c = CHUNKS[section_id]
    pages = f"p. {c['page_start']}" if c["page_start"] == c["page_end"] else f"pp. {c['page_start']}-{c['page_end']}"
    return f"[{section_id}] ({c['doc']}, {pages})\n{c['text']}"


print(f"✓ Run: {RUN} ({'smoke test' if INFO['smoke_test'] else 'full run'}, {len(INFO['scenes'])} scenes)")
for w in WRITERS:
    i = INFO["writers"][w]
    print(f"✓ {w}: {i['repo']} at {i['commit'][:12]}, server overrides {i['server_overrides']}, "
          f"loaded in {i['load_minutes']} min, scenes in {i['scenes_minutes']} min")
print(f"✓ {len(CHUNKS)} guideline sections in the index")

✓ Run: D:\CSE499AB_project\Essential_task_delegation\Offline_RAG_KR2\results\t2b\t2b_full_internet_on (full run, 10 scenes)
✓ 7b: AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP at def8f6c81889, server overrides None, loaded in 4.0 min, scenes in 62.3 min
✓ 3b-instruct: Qwen/Qwen2.5-3B-Instruct at aa8e72537993, server overrides {'top_p': 1.0, 'top_k': -1}, loaded in 1.8 min, scenes in 180.1 min
✓ 405 guideline sections in the index


In [2]:
# ── Cell 2: Each report's sources (as step 7), and every count per report ───────────
def between(text, start, end="\n\nReport: "):
    i = text.index(start) + len(start)
    return text[i:text.index(end, i)].strip()


def report_sources(scene_dir):
    """{report: the facts it was given (damage facts, map reading, passages)}, and the search IDs per report."""
    log = (scene_dir / "run_log.txt").read_text(encoding="utf-8")
    facts = between(log, "C4's damage facts, given to every report:\n")
    reading = between(log, "Map reading made by the code from the hot spots:\n")
    searched, pending = {}, []
    for rec in map(json.loads, open(scene_dir / "llm_calls.jsonl", encoding="utf-8")):
        if rec["kind"] == "search":
            pending += rec["ids"]
        elif rec.get("report") and pending:
            searched.setdefault(rec["report"], []).extend(pending)
            pending = []
    ems98 = list(dict.fromkeys(searched.get("Expert_team_report.txt", [])))
    out, shown = {}, {}
    for report, spec in SPECS.items():
        ids = list(dict.fromkeys(searched.get(report, []) + (ems98 if spec["ems98"] else [])))
        parts = [("damage facts for the photo pair (from the system's damage model; estimates)", facts)]
        if spec["map"]:
            parts.append(("map reading (made by the code from the damage model's hot spots and safe areas)", reading))
        if ids:
            parts.append(("guideline sections (from the guideline search; cite these IDs)", "\n\n".join(map(passage, ids))))
        out[report] = "\n\n".join(f"=== Facts: {t} ===\n{b}" for t, b in parts)
        shown[report] = set(ids)
    return out, shown


def items(problems, kinds):
    """Tokens of the given kinds in a check record's problem list ("number_wrong: 96%, 12")."""
    found = []
    for p in problems or []:
        kind, _, rest = p.partition(": ")
        if kind in kinds and rest:
            found += rest.split(", ")
    return found


rows, flagged_rows = [], []
for w in WRITERS:
    for scene in INFO["scenes"]:
        scene_dir = RUN / w / f"run_{Path(scene).stem}"
        summary = json.load(open(scene_dir / "t2_run_summary.json", encoding="utf-8"))
        scene_passes = all(summary["rules"].values())
        sources, shown = report_sources(scene_dir)
        calls = [json.loads(l) for l in open(scene_dir / "llm_calls.jsonl", encoding="utf-8")]
        for report, spec in SPECS.items():
            path = scene_dir / "reports" / "offline" / report
            text = path.read_text(encoding="utf-8") if path.is_file() else ""
            parts = re.split(r"^## (.+)$", text, flags=re.MULTILINE)
            sections = {h.strip(): b.strip() for h, b in zip(parts[1::2], parts[2::2])}
            model_sections = {h: b for h, b in sections.items() if h != GRADES_HEADING}
            checks = [c for c in calls if c["kind"] == "check" and c.get("report") == report]
            guards = [c for c in calls if c["kind"] == "number_guard" and c.get("report") == report]
            k = nc.known(*nc.sources_of(sources[report]))
            final = [n for b in model_sections.values() for n in nc.check(b, k, version=2)]
            for c in checks:
                for kind in ("number_wrong", "number_misused", "number_unsourced"):
                    for token in items(c.get("problems_left"), {kind}):
                        flagged_rows.append({"writer": w, "scene": scene, "report": report, "section": c["section"],
                                             "token": token, "verdict": kind[7:], "where": "left after asking again"})
            for n in final:
                if n["verdict"] != "ok":
                    flagged_rows.append({"writer": w, "scene": scene, "report": report, "section": "?",
                                         "token": n["token"], "verdict": n["verdict"], "where": "finished report",
                                         "sentence": n["sentence"]})
            cited = [i for i in BRACKETED.findall("\n".join(model_sections.values())) if i.startswith(ID_PREFIXES)]
            for earlier in spec["uses"]:                    # IDs in the earlier reports it was given count as shown
                p = scene_dir / "reports" / "offline" / earlier
                if p.is_file():
                    shown[report] |= set(BRACKETED.findall(p.read_text(encoding="utf-8")))
            task = next((c for c in calls if c["kind"] == "task" and c.get("report") == report), {})
            rows.append({
                "writer": w, "scene": scene, "report": report,
                "finished": bool(text) and scene_passes and len(sections) >= spec.get("sections", 1)
                            and not any(b.startswith("An error occurred") for b in sections.values()),
                "sections": len(sections),
                "words": sum(len(b.split()) for b in model_sections.values()),
                "numbers_checked": len(final),
                "contradictions": sum(len(items(c.get("problems_left"), {"number_wrong", "number_misused"})) for c in checks)
                                  + sum(n["verdict"] in ("wrong", "misused") for n in final),
                "unsourced": sum(len(items(c.get("problems_left"), {"number_unsourced"})) for c in checks)
                             + sum(n["verdict"] == "unsourced" for n in final),
                "guard_replaced": sum(len(g.get("replaced", [])) for g in guards),
                "first_answer_number_problems": sum(len(items(c.get("problems"), {"number_wrong", "number_misused",
                                                                                   "number_unsourced"})) for c in checks),
                "asked_again": sum(bool(c.get("asked_again")) for c in checks),
                "fake_citations": sum(i not in CHUNKS for i in cited),
                "real_citations": sum(i in CHUNKS and i in shown[report] for i in cited),
                "real_not_shown": sum(i in CHUNKS and i not in shown[report] for i in cited),
                "not_available_offline": len(NOT_AVAILABLE.findall(text)) - text.count(GUARD),   # the writer's own, not the guard's
                "seconds": task.get("seconds"),
            })
REPORTS = pd.DataFrame(rows)
FLAGGED = pd.DataFrame(flagged_rows)
print(REPORTS.groupby("writer")[["finished", "contradictions", "unsourced", "fake_citations", "real_citations",
                                  "guard_replaced", "asked_again", "words"]].sum().to_string())

             finished  contradictions  unsourced  fake_citations  real_citations  guard_replaced  asked_again  words
writer                                                                                                              
3b-instruct        60              15         12               0             604              35           73  71124
7b                 60               3          1               3             246               5           56  27805


In [3]:
# ── Cell 3: The rule, the expectation, and the measures beside them ─────────────────
T = REPORTS.groupby("writer").agg(reports=("report", "size"), finished=("finished", "sum"),
                                  contradictions=("contradictions", "sum"), unsourced=("unsourced", "sum"),
                                  fake_citations=("fake_citations", "sum"), real_not_shown=("real_not_shown", "sum"),
                                  real_citations=("real_citations", "sum"), guard_replaced=("guard_replaced", "sum"),
                                  first_answer_number_problems=("first_answer_number_problems", "sum"),
                                  sections_asked_again=("asked_again", "sum"),
                                  not_available_offline=("not_available_offline", "sum"),
                                  words_per_report=("words", "median"))
def section_words(w):
    """Words in every model-written section of the writer's reports."""
    out = []
    for s in INFO["scenes"]:
        for r in SPECS:
            path = RUN / w / f"run_{Path(s).stem}" / "reports" / "offline" / r
            parts = re.split(r"^## (.+)$", path.read_text(encoding="utf-8"), flags=re.M) if path.is_file() else []
            out += [len(b.split()) for h, b in zip(parts[1::2], parts[2::2]) if h.strip() != GRADES_HEADING]
    return out


T["words_per_section_median"] = [pd.Series(section_words(w)).median() for w in T.index]
T["minutes_per_scene"] = [round(INFO["writers"][w]["scenes_minutes"] / len(INFO["scenes"]), 2) for w in T.index]
T["memory_lines"] = ["; ".join(l.split("]")[-1].strip() for l in INFO["writers"][w]["server_log_lines"]
                               if "Model loading took" in l)[:120] for w in T.index]
print(T.T.to_string())

big, small = "7b", "3b-instruct"
a, b = T.loc[big], T.loc[small]
COLS = {"no more contradictions": "contradictions", "no more fake citations": "fake_citations",
        "finishes at least as many reports": "finished"}
tests = {"no more contradictions": b["contradictions"] <= a["contradictions"],
         "no more fake citations": b["fake_citations"] <= a["fake_citations"],
         "finishes at least as many reports": b["finished"] >= a["finished"]}
TEST_LINES = [f"{'✓' if v else '✗'} the text model {k}: 7B {int(a[COLS[k]])}, 3B {int(b[COLS[k]])}"
              for k, v in tests.items()]
VERDICT = ("the text model passes the rule: it can replace the 7B as the writer, with the supervisor's agreement"
           if all(tests.values()) else "the 7B stays the writer")
print("\n".join(TEST_LINES))
print("Rule:", VERDICT)

n = len(INFO["scenes"]) * len(SPECS)
EXPECT = [
    ("both writers finish every report", f"{n} each", f"7B {int(a['finished'])}, 3B {int(b['finished'])}",
     a["finished"] == n and b["finished"] == n),
    ("contradictions at most 3 per writer", "≤ 3 each", f"7B {int(a['contradictions'])}, 3B {int(b['contradictions'])}",
     a["contradictions"] <= 3 and b["contradictions"] <= 3),
    ("fake citations at most 2 per writer", "≤ 2 each", f"7B {int(a['fake_citations'])}, 3B {int(b['fake_citations'])}",
     a["fake_citations"] <= 2 and b["fake_citations"] <= 2),
    ("the text model's sections are longer", "median ≥ 1.2 × the 7B's",
     f"7B {a['words_per_section_median']:.0f}, 3B {b['words_per_section_median']:.0f} words",
     b["words_per_section_median"] >= 1.2 * a["words_per_section_median"]),
    ("the text model is faster", "≤ 0.7 × the 7B's minutes per scene",
     f"7B {a['minutes_per_scene']}, 3B {b['minutes_per_scene']} min", b["minutes_per_scene"] <= 0.7 * a["minutes_per_scene"]),
]
EXPECTATION = pd.DataFrame(EXPECT, columns=["expected", "how much", "result", "met"])
print(EXPECTATION.to_string(index=False))

writer                                                                                                               3b-instruct                                                                                                  7b
reports                                                                                                                       60                                                                                                  60
finished                                                                                                                      60                                                                                                  60
contradictions                                                                                                                15                                                                                                   3
unsourced                                                                           

In [4]:
# ── Cell 4: Every flagged number, the blind-reading pairs, and the files ──────────────
def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


print(FLAGGED.to_string(index=False) if len(FLAGGED) else "(no number flagged for either writer)")

# Five (scene, report) slots, drawn with a fixed seed before the run; each slot's two reports as A and B in a drawn
# order, the key kept apart. The mode line (which says nothing about the writer) is left out of what is read.
rng = random.Random(20261121)
slots = rng.sample([(s, r) for s in INFO["scenes"] for r in SPECS], 5)
blind = OUT / "blind"
blind.mkdir(parents=True, exist_ok=True)
key = {}
for i, (scene, report) in enumerate(slots, 1):
    order = rng.sample(WRITERS, 2)
    key[f"pair {i}"] = {"scene": scene, "report": report, "A": order[0], "B": order[1]}
    for label, w in zip("AB", order):
        text = (RUN / w / f"run_{Path(scene).stem}" / "reports" / "offline" / report).read_text(encoding="utf-8")
        (blind / f"pair_{i}_{label}.txt").write_text(text.split("\n", 2)[2] if text.startswith("Mode:") else text,
                                                     encoding="utf-8")
(blind / "key_do_not_open_before_reading.json").write_text(json.dumps(key, indent=1), encoding="utf-8")
(blind / "README.md").write_text("\n".join([
    "# T2b: five pairs to read blind", "",
    "Each pair is the same report for the same scene, written by the two writers; which one is A is drawn at random.",
    "Read both, then note for each pair which is more useful to a responder (A, B or the same), and why, before",
    "opening `key_do_not_open_before_reading.json`. This reading is reported beside the rule; it doesn't decide it.", "",
    *[f"- pair {i}: {k['report'][:-4].replace('_', ' ')}, scene `{k['scene']}`" for i, k in
      enumerate(key.values(), 1)]]) + "\n", encoding="utf-8")

REPORTS.to_csv(OUT / "reports.csv", index=False)
FLAGGED.to_csv(OUT / "flagged_numbers.csv", index=False)
T.to_csv(OUT / "writers.csv")
(OUT / "scores.md").write_text("\n".join([
    "# T2b: the two writers, scored", "",
    f"Made by [t2b_score.ipynb](../../t2b_score.ipynb) from `{RUN.name}`. How each count is made is in its first cell.", "",
    f"**Rule:** {VERDICT}.", "",
    *[f"- {line}" for line in TEST_LINES], "",
    "## Per writer", "", md(T.reset_index()), "",
    "## Expectation, fixed before the run, against the result", "", md(EXPECTATION), "",
    "## Every number flagged", "", md(FLAGGED) if len(FLAGGED) else "None.", "",
    "Files: `reports.csv` (one row per writer, scene and report), `flagged_numbers.csv`, `writers.csv`, `blind/`."])
    + "\n", encoding="utf-8")
print(f"✓ Wrote scores.md, reports.csv, flagged_numbers.csv, writers.csv and blind/ (5 pairs) to {OUT}")

     writer                                          scene                                          report                             section             token   verdict                   where
         7b                   beriut_explosion_post_16.png                         Reconstruction_plan.txt                      Plan of Action               3.5 unsourced left after asking again
         7b        palu_tsunami_00000180_post_disaster.png                         Reconstruction_plan.txt                Assessment of Damage               49%     wrong left after asking again
         7b        palu_tsunami_00000180_post_disaster.png                         Reconstruction_plan.txt                Assessment of Damage               33%     wrong left after asking again
         7b  tuscaloosa_tornado_00000037_post_disaster.png                          Expert_team_report.txt                            Overview               54%     wrong left after asking again
3b-instruct              